# Employee Attrition Analyzer
### Lab 7 — Open Ended Lab (Machine Learning)

| | |
|---|---|
| **Project Name** | Employee Attrition Analyzer |
| **Name** | M. Hassan |
| **Roll No** | 24F-AI-065 |
| **Dataset** | IBM HR Analytics Employee Attrition & Performance |

**Objective:** Analyze employee records and build machine learning models that predict whether an employee is likely to leave the company (Attrition = Yes/No).

## 1. Dataset Description

The **IBM HR Analytics Employee Attrition & Performance** dataset contains **1470 employee records** with **35 features** describing each employee, such as:

- **Demographics:** Age, Gender, MaritalStatus, Education, EducationField
- **Job-related:** Department, JobRole, JobLevel, JobSatisfaction, JobInvolvement, OverTime, BusinessTravel
- **Compensation:** MonthlyIncome, DailyRate, HourlyRate, PercentSalaryHike, StockOptionLevel
- **Experience:** TotalWorkingYears, YearsAtCompany, YearsInCurrentRole, NumCompaniesWorked
- **Target variable:** `Attrition` — "Yes" (employee left) or "No" (employee stayed)

The dataset is **imbalanced**: ~84% employees stayed and only ~16% left, which is typical for attrition problems.

## 2. Importing Required Libraries

In [1]:
# -------------------------------------------------------------------
# Project Name : Employee Attrition Analyzer
# Name         : M. Hassan
# Roll No      : 24F-AI-065
# Description  : Predicting employee attrition using ML models
#                (Logistic Regression, Decision Tree, Random Forest)
# -------------------------------------------------------------------

# Data handling libraries
import numpy as np
import pandas as pd

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing tools
from sklearn.model_selection import train_test_split          # to split data into train/test sets
from sklearn.preprocessing import LabelEncoder, StandardScaler  # encoding + scaling

# Machine Learning models
from sklearn.linear_model import LogisticRegression    # Model 1
from sklearn.tree import DecisionTreeClassifier        # Model 2
from sklearn.ensemble import RandomForestClassifier    # Model 3

# Evaluation metrics
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

sns.set_style("whitegrid")   # nicer looking plots
print("All libraries imported successfully!")

All libraries imported successfully!


## 3. Loading the Dataset

In [2]:
# Load the CSV file into a pandas DataFrame
df = pd.read_csv("WA_Fn-UseC_-HR-Employee-Attrition.csv")

# Display first 5 rows to see what the data looks like
df.head()

   Age Attrition     BusinessTravel  DailyRate              Department  \
0   41       Yes      Travel_Rarely       1102                   Sales   
1   49        No  Travel_Frequently        279  Research & Development   
2   37       Yes      Travel_Rarely       1373  Research & Development   
3   33        No  Travel_Frequently       1392  Research & Development   
4   27        No      Travel_Rarely        591  Research & Development   

   DistanceFromHome  Education EducationField  EmployeeCount  EmployeeNumber  \
0                 1          2  Life Sciences              1               1   
1                 8          1  Life Sciences              1               2   
2                 2          2          Other              1               4   
3                 3          4  Life Sciences              1               5   
4                 2          1        Medical              1               7   

   ...  RelationshipSatisfaction StandardHours  StockOptionLevel  \
0  ...

In [3]:
# Basic information about the dataset
print("Shape of dataset (rows, columns):", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nData types and non-null counts:")
df.info()

Shape of dataset (rows, columns): (1470, 35)

Column names:
['Age', 'Attrition', 'BusinessTravel', 'DailyRate', 'Department', 'DistanceFromHome', 'Education', 'EducationField', 'EmployeeCount', 'EmployeeNumber', 'EnvironmentSatisfaction', 'Gender', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobRole', 'JobSatisfaction', 'MaritalStatus', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'Over18', 'OverTime', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StandardHours', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']

Data types and non-null counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Age                       1470 non-null   int64 
 1   Attrition     

In [4]:
# Check for missing values in each column
print("Total missing values in dataset:", df.isnull().sum().sum())

# Check the distribution of the target variable (Attrition)
print("\nTarget variable distribution:")
print(df["Attrition"].value_counts())
print("\nPercentage:")
print((df["Attrition"].value_counts(normalize=True) * 100).round(2))

Total missing values in dataset: 0

Target variable distribution:
Attrition
No     1233
Yes     237
Name: count, dtype: int64

Percentage:
Attrition
No     83.88
Yes    16.12
Name: proportion, dtype: float64


## 4. Exploratory Data Analysis (EDA) & Visualization

Before preprocessing, we visualize the data to understand patterns related to attrition.

In [5]:
# --- Plot 1: Attrition count ---
plt.figure(figsize=(5, 4))
sns.countplot(x="Attrition", data=df, palette="Set2")
plt.title("Employee Attrition Distribution")
plt.ylabel("Number of Employees")
plt.show()
# Observation: the dataset is imbalanced - many more "No" than "Yes"

In [6]:
# --- Plot 2: Attrition by Department ---
plt.figure(figsize=(8, 4))
sns.countplot(x="Department", hue="Attrition", data=df, palette="Set1")
plt.title("Attrition by Department")
plt.xticks(rotation=15)
plt.show()

In [7]:
# --- Plot 3: Attrition by OverTime ---
plt.figure(figsize=(5, 4))
sns.countplot(x="OverTime", hue="Attrition", data=df, palette="Set3")
plt.title("Attrition by OverTime")
plt.show()
# Observation: employees working overtime leave much more often

In [8]:
# --- Plot 4: Age distribution for Attrition groups ---
plt.figure(figsize=(7, 4))
sns.histplot(data=df, x="Age", hue="Attrition", kde=True, bins=30, palette="husl")
plt.title("Age Distribution by Attrition")
plt.show()

## 5. Data Preprocessing

Preprocessing steps applied:
1. **Drop irrelevant/constant columns** — `EmployeeCount`, `Over18`, `StandardHours` have the same value for every employee, and `EmployeeNumber` is just an ID, so they carry no predictive information.
2. **Encode the target variable** — convert `Attrition` from Yes/No to 1/0.
3. **Encode categorical features** — convert text columns (Department, JobRole, Gender, etc.) into numeric form using one-hot encoding (`pd.get_dummies`).
4. **Train/Test split** — 80% training data, 20% testing data.
5. **Feature scaling** — standardize numeric columns so all features are on the same scale (important for Logistic Regression).

In [9]:
# Step 1: Drop columns that are useless for prediction
# EmployeeCount = always 1, Over18 = always "Y", StandardHours = always 80,
# EmployeeNumber = just an ID number
df = df.drop(["EmployeeCount", "Over18", "StandardHours", "EmployeeNumber"], axis=1)
print("Columns after dropping:", df.shape[1])

Columns after dropping: 31


In [10]:
# Step 2: Encode the target variable Attrition: Yes -> 1, No -> 0
le = LabelEncoder()
df["Attrition"] = le.fit_transform(df["Attrition"])
print("Attrition encoded (1 = Yes, 0 = No):")
print(df["Attrition"].value_counts())

Attrition encoded (1 = Yes, 0 = No):
Attrition
0    1233
1     237
Name: count, dtype: int64


In [11]:
# Step 3: One-hot encode all remaining categorical (text) columns
# get_dummies converts each category into its own 0/1 column
categorical_cols = df.select_dtypes(include=["object"]).columns
print("Categorical columns to encode:", list(categorical_cols))

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
print("\nShape after encoding:", df_encoded.shape)

Categorical columns to encode: ['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime']

Shape after encoding: (1470, 45)


In [12]:
# Step 4: Separate features (X) and target (y)
X = df_encoded.drop("Attrition", axis=1)   # all columns except the target
y = df_encoded["Attrition"]                # the column we want to predict

# Step 5: Split into training (80%) and testing (20%) sets
# stratify=y keeps the same Yes/No ratio in both sets
# random_state=42 makes the split reproducible
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print("Training set size:", X_train.shape)
print("Testing set size :", X_test.shape)

Training set size: (1176, 44)
Testing set size : (294, 44)


In [13]:
# Step 6: Feature scaling using StandardScaler (mean=0, std=1)
# We fit the scaler ONLY on training data to avoid data leakage,
# then use it to transform both training and testing data.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling complete! Example of scaled values (first row, first 5 features):")
print(X_train_scaled[0][:5])

Scaling complete! Example of scaled values (first row, first 5 features):
[ 1.09019402  1.04945488 -0.89991452  1.06420933 -0.65870975]


## 6. Machine Learning Models

Three classification models are trained and compared:

1. **Logistic Regression** — a simple linear model that estimates the probability of attrition. It needs scaled data, so we give it the scaled features.
2. **Decision Tree** — splits the data into a tree of if/else rules. Easy to interpret. Trees do not need scaling.
3. **Random Forest** — an ensemble of many decision trees whose predictions are combined (majority vote). Usually more accurate and less overfitting than a single tree.

**Integration:** all three models are trained on the *same* preprocessed training set and evaluated on the *same* test set, so their results are directly comparable.

In [14]:
# ---------- Model 1: Logistic Regression ----------
# max_iter increased so the model has enough iterations to converge
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)              # train on SCALED data
y_pred_lr = log_reg.predict(X_test_scaled)        # predict on scaled test data
print("Logistic Regression trained successfully!")

Logistic Regression trained successfully!


In [15]:
# ---------- Model 2: Decision Tree ----------
# max_depth limits tree growth to reduce overfitting
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)                          # trees don't need scaled data
y_pred_dt = dt.predict(X_test)
print("Decision Tree trained successfully!")

Decision Tree trained successfully!


In [16]:
# ---------- Model 3: Random Forest ----------
# n_estimators = number of trees in the forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print("Random Forest trained successfully!")

Random Forest trained successfully!


## 7. Model Evaluation

Each model is evaluated using:
- **Accuracy** — overall percentage of correct predictions
- **Precision** — of the employees predicted to leave, how many actually left
- **Recall** — of the employees who actually left, how many did we catch
- **F1 Score** — harmonic mean of precision and recall (best single metric for imbalanced data)
- **Confusion Matrix** — table of correct vs incorrect predictions

In [17]:
# Store model names and their predictions in a dictionary for easy comparison
models = {
    "Logistic Regression": y_pred_lr,
    "Decision Tree": y_pred_dt,
    "Random Forest": y_pred_rf
}

# Calculate evaluation metrics for every model and put them in a table
results = []
for name, y_pred in models.items():
    results.append({
        "Model": name,
        "Accuracy":  accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall":    recall_score(y_test, y_pred),
        "F1 Score":  f1_score(y_test, y_pred)
    })

results_df = pd.DataFrame(results).round(3)
print("Model Comparison Table:")
results_df

Model Comparison Table:


                 Model  Accuracy  Precision  Recall  F1 Score
0  Logistic Regression     0.861      0.615   0.340     0.438
1        Decision Tree     0.833      0.444   0.170     0.246
2        Random Forest     0.833      0.417   0.106     0.169

In [18]:
# --- Confusion Matrix for each model ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, y_pred) in zip(axes, models.items()):
    cm = confusion_matrix(y_test, y_pred)          # build confusion matrix
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Stay (0)", "Leave (1)"],
                yticklabels=["Stay (0)", "Leave (1)"])
    ax.set_title(f"Confusion Matrix\n{name}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.show()

In [19]:
# --- Bar chart comparing the models on all metrics ---
results_df.set_index("Model").plot(kind="bar", figsize=(9, 5))
plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=15)
plt.legend(loc="lower right")
plt.show()

In [20]:
# Detailed classification report for the best performing model (Logistic Regression)
print("Classification Report - Logistic Regression:")
print(classification_report(y_test, y_pred_lr, target_names=["Stay", "Leave"]))

Classification Report - Logistic Regression:
              precision    recall  f1-score   support

        Stay       0.88      0.96      0.92       247
       Leave       0.62      0.34      0.44        47

    accuracy                           0.86       294
   macro avg       0.75      0.65      0.68       294
weighted avg       0.84      0.86      0.84       294



## 8. Feature Importance Visualization

Random Forest can tell us **which features matter most** for predicting attrition. This helps HR understand the main drivers of employees leaving.

In [21]:
# Get feature importances from the trained Random Forest model
importances = rf.feature_importances_
feature_names = X.columns

# Put them into a DataFrame and sort from most to least important
feat_imp = pd.DataFrame({"Feature": feature_names, "Importance": importances})
feat_imp = feat_imp.sort_values("Importance", ascending=False)

# Plot the top 15 most important features
plt.figure(figsize=(9, 6))
sns.barplot(x="Importance", y="Feature", data=feat_imp.head(15), palette="viridis")
plt.title("Top 15 Feature Importances (Random Forest)")
plt.tight_layout()
plt.show()

## 9. Conclusion

- The IBM HR Analytics dataset (1470 employees, 35 features) was cleaned, encoded, scaled, and split into training (80%) and testing (20%) sets.
- Three ML models were trained and compared on the same data: **Logistic Regression**, **Decision Tree**, and **Random Forest**.
- In this run, **Logistic Regression** gave the best balance — the highest accuracy (~86%) and the best F1 score and recall for the "Leave" class. The tree-based models achieved good accuracy but missed more of the employees who actually left.
- Because the data is imbalanced (only ~16% leave), **F1 score and recall** are more meaningful than accuracy alone — a model that always predicts "Stay" would already get ~84% accuracy while being useless.
- Feature importance shows the strongest drivers of attrition are **OverTime, MonthlyIncome, Age, TotalWorkingYears and YearsAtCompany** — employees who work overtime, earn less, and are younger/less experienced are the most likely to leave.
- HR can use these insights to target retention efforts at high-risk employee groups.